# 🌤️ AEMET Weather AI Assistant — Unified Chain (Phase 1 + Phase 2)

This notebook merges:
- **Phase 1** → Live weather data from AEMET API + LLM (Groq / LLaMA)
- **Phase 2** → RAG pipeline over AEMET alert PDFs (ChromaDB + HuggingFace embeddings)

**Unified chain flow:**
```
User question
      │
      ├─► [Phase 1] AEMET API → live forecast (Madrid, Barcelona, ...)
      │
      ├─► [Phase 2] ChromaDB retriever → relevant alert/context chunks
      │
      └─► Combined prompt → ChatGroq (LLaMA) → Final answer
```

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-groq \
               langchain-text-splitters huggingface_hub sentence-transformers \
               chromadb pypdf requests pydantic groq

In [ ]:
!pip install langdetect -q

In [ ]:
!pip install langchain-chroma -q

In [ ]:
!pip install -q --upgrade langchain langchain-core langchain-groq

In [ ]:
!pip install ipywidgets

## 2. API Keys

In [ ]:
import os

AEMET_API_KEY = os.getenv("AEMET_API_KEY", "YOUR_AEMET_API_KEY_HERE")
HF_TOKEN    = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
GROQ_API_KEY= os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

print("Keys loaded:", bool(AEMET_API_KEY), bool(HF_TOKEN), bool(GROQ_API_KEY))

> **⚠️ Security Note:** Never hardcode API keys. Set them as environment variables:
> ```bash
> export AEMET_API_KEY="your_key_here"
> export HF_TOKEN="your_token_here"
> export GROQ_API_KEY="your_key_here"
> ```
> Or use a `.env` file with `python-dotenv` (add `.env` to `.gitignore`).

---
## ☁️ PHASE 1 — Live Weather from AEMET API

### 3. AEMET API Client

In [ ]:
import requests

class AEMETClient:
    BASE_URL = "https://opendata.aemet.es/opendata/api"

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.headers = {"accept": "application/json"}
        self.params  = {"api_key": self.api_key}

    def _fetch(self, endpoint: str) -> dict:
        """AEMET two-step fetch: first call returns a data URL, second call returns actual data."""
        url = f"{self.BASE_URL}{endpoint}"
        r   = requests.get(url, headers=self.headers, params=self.params)
        r.raise_for_status()
        response = r.json()
        data_url = response.get("datos")
        if not data_url:
            return {}
        data_r = requests.get(data_url, headers=self.headers)
        data_r.raise_for_status()
        return data_r.json()

    def get_weather_by_city(self, municipio_code: str) -> dict:
        """Get hourly forecast for a Spanish municipality code."""
        endpoint = f"/prediccion/especifica/municipio/horaria/{municipio_code}"
        data = self._fetch(endpoint)
        return data[0] if data else {}

print("AEMETClient defined ✅")

### 4. Forecast Parser

In [ ]:
def parse_daily_forecast(day: dict) -> dict:
    """Convert raw AEMET day data into a clean LLM-friendly summary."""
    temps = [int(t["value"]) for t in day.get("temperatura", []) if t.get("value")]
    sky   = list(set([s["descripcion"] for s in day.get("estadoCielo", []) if s.get("descripcion")]))
    rain  = [f"Period {r['periodo']}: {r['value']}%" for r in day.get("probPrecipitacion", []) if r.get("value")]
    storm = [f"Period {s['periodo']}: {s['value']}%" for s in day.get("probTormenta", [])    if s.get("value")]
    return {
        "date":            day.get("fecha", "")[:10],
        "temp_min":        min(temps) if temps else None,
        "temp_max":        max(temps) if temps else None,
        "sky_conditions":  sky,
        "rain_probability":rain,
        "storm_probability":storm,
        "sunrise":         day.get("orto"),
        "sunset":          day.get("ocaso"),
    }

def get_forecast_summary(weather_data: dict) -> list:
    """Parse all available days from an AEMET city response."""
    days = weather_data.get("prediccion", {}).get("dia", [])
    return [parse_daily_forecast(day) for day in days]

print("Forecast parsers defined ✅")

### Fetch live forecast of Spain(52 cities)

In [ ]:
import time
import requests
from datetime import datetime, timezone, date

SPAIN_CITIES = {
    # Andalucía
    "Almería":       "04013",
    "Cádiz":         "11012",
    "Córdoba":       "14021",
    "Granada":       "18087",
    "Huelva":        "21041",
    "Jaén":          "23050",
    "Málaga":        "29067",
    "Sevilla":       "41091",
    # Aragón
    "Huesca":        "22125",
    "Teruel":        "44216",
    "Zaragoza":      "50297",
    # Asturias
    "Oviedo":        "33044",
    # Baleares
    "Palma":         "07040",
    # Canarias
    "Las Palmas":    "35016",
    "Santa Cruz":    "38038",
    # Cantabria
    "Santander":     "39075",
    # Castilla-La Mancha
    "Albacete":      "02003",
    "Ciudad Real":   "13034",
    "Cuenca":        "16078",
    "Guadalajara":   "19130",
    "Toledo":        "45168",
    # Castilla y León
    "Ávila":         "05019",
    "Burgos":        "09059",
    "León":          "24089",
    "Palencia":      "34120",
    "Salamanca":     "37274",
    "Segovia":       "40155",
    "Soria":         "42173",
    "Valladolid":    "47186",
    "Zamora":        "49275",
    # Cataluña
    "Barcelona":     "08019",
    "Girona":        "17079",
    "Lleida":        "25120",
    "Tarragona":     "43148",
    # Extremadura
    "Badajoz":       "06015",
    "Cáceres":       "10037",
    # Galicia
    "A Coruña":      "15030",
    "Lugo":          "27028",
    "Ourense":       "32054",
    "Pontevedra":    "36038",
    # La Rioja
    "Logroño":       "26089",
    # Madrid
    "Madrid":        "28079",
    # Murcia
    "Murcia":        "30030",
    # Navarra
    "Pamplona":      "31201",
    # País Vasco
    "Vitoria":       "01059",
    "San Sebastián": "20069",
    "Bilbao":        "48020",
    # Valencia
    "Alicante":      "03014",
    "Castellón":     "12040",
    "Valencia":      "46250",
    # Ceuta & Melilla
    "Ceuta":         "51001",
    "Melilla":       "52001",
}


def normalize(text: str) -> str:
    """Lowercase + remove accents for fuzzy matching."""
    return (text.lower()
            .replace("á","a").replace("é","e").replace("í","i")
            .replace("ó","o").replace("ú","u").replace("ü","u")
            .replace("ñ","n"))


def validate_and_fix_codes(api_key: str, cities: dict) -> dict:
    """
    Tests every code against AEMET. If invalid, searches the master
    municipality list for the correct code automatically.
    Returns a dict of {city: verified_code}.
    """
    BASE_URL = "https://opendata.aemet.es/opendata/api"
    headers  = {"accept": "application/json"}
    params   = {"api_key": api_key}
    valid    = {}
    invalid  = {}

    print("🔍 Validating municipality codes...\n")
    for city, code in cities.items():
        url = f"{BASE_URL}/prediccion/especifica/municipio/horaria/{code}"
        try:
            r      = requests.get(url, headers=headers, params=params, timeout=10)
            data   = r.json()
            estado = data.get("estado", 0)
            if estado == 200:
                valid[city] = code
                print(f"  ✅ {city} ({code})")
            else:
                desc = data.get("descripcion", "Unknown error")
                print(f"  ❌ {city} ({code}) — status {estado}: {desc}")
                invalid[city] = {"bad_code": code, "reason": desc}
        except Exception as e:
            print(f"  ❌ {city} ({code}) — {e}")
            invalid[city] = {"bad_code": code, "reason": str(e)}
        time.sleep(0.3)

    # ── Auto-fix invalid codes using AEMET master list ────────────────────────
    if invalid:
        print(f"\n🔧 Fetching AEMET master municipality list to fix {len(invalid)} cities...")
        all_munis = []
        try:
            r        = requests.get(f"{BASE_URL}/maestro/municipios",
                                    headers=headers, params=params, timeout=10)
            data_url = r.json().get("datos")
            if data_url:
                r2        = requests.get(data_url, headers=headers, timeout=15)
                all_munis = r2.json()
                print(f"   Master list loaded: {len(all_munis)} municipalities\n")
        except Exception as e:
            print(f"   ⚠️ Could not load master list: {e}")

        for city, info in invalid.items():
            if not all_munis:
                print(f"  ⚠️  {city} → master list unavailable, skipping")
                continue
            city_norm = normalize(city)
            # Find all municipalities whose name contains the city name
            matches = [
                m for m in all_munis
                if city_norm in normalize(m.get("nombre", ""))
            ]
            if matches:
                # Prefer exact match first, otherwise take first partial match
                exact = [m for m in matches if normalize(m.get("nombre","")) == city_norm]
                best  = exact[0] if exact else matches[0]

                # ── FIX: AEMET ID format is "id28079" — strip the "id" prefix safely
                raw_id   = best.get("id", "")
                new_code = raw_id[2:] if raw_id.startswith("id") else raw_id

                valid[city] = new_code
                print(f"  ✅ {city} → corrected code: {new_code} "
                      f"(matched: '{best.get('nombre')}', was: {info['bad_code']})")
            else:
                print(f"  ⚠️  {city} → no match in master list, skipping")
            time.sleep(0.2)

    print(f"\n  Valid: {len(valid)}/{len(cities)} codes confirmed")
    if len(valid) < len(cities):
        print(f"  Skipped: {set(cities) - set(valid)}")
    return valid


def fetch_city(client, city: str, code: str, delay: float = 0.5) -> tuple:
    time.sleep(delay)
    raw      = client.get_weather_by_city(code)
    forecast = get_forecast_summary(raw)
    if not forecast:
        raise ValueError("Empty forecast returned")
    return city, forecast


def fetch_all_spain(
    api_key:       str,
    delay_seconds: float = 0.5,
    max_retries:   int   = 3,
    retry_delay:   float = 5.0
) -> tuple:
    """
    Full fetch pipeline:
      Step 0 — validate/fix all municipality codes
      Step 1 — fetch all verified cities
      Step 2 — retry failures with exponential backoff
      Step 3 — top-up pass for any still-missing cities
    """
    client     = AEMETClient(api_key)
    fetch_time = datetime.now(timezone.utc)
    forecasts  = {}
    failed     = {}

    # ── Step 0: Validate codes ────────────────────────────────────────────────
    verified_cities = validate_and_fix_codes(api_key, SPAIN_CITIES)
    print()

    # ── Step 1: First pass ────────────────────────────────────────────────────
    print(f"📡 Fetching forecasts for {len(verified_cities)} cities...")
    print(f"   Started (UTC): {fetch_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    for i, (city, code) in enumerate(verified_cities.items(), 1):
        try:
            _, forecast     = fetch_city(client, city, code, delay_seconds)
            forecasts[city] = forecast
            print(f"  [{i:02d}/{len(verified_cities)}] ✅ {city}")
        except Exception as e:
            print(f"  [{i:02d}/{len(verified_cities)}] ❌ {city} — {e}")
            failed[city] = str(e)

    # ── Step 2: Retry with exponential backoff ────────────────────────────────
    for retry_round in range(1, max_retries + 1):
        if not failed:
            break
        print(f"\n🔄 Retry {retry_round}/{max_retries} — "
              f"{len(failed)} cities: {list(failed.keys())}")
        print(f"   Waiting {retry_delay}s...\n")
        time.sleep(retry_delay)

        still_failed = {}
        for city in list(failed.keys()):
            code = verified_cities[city]
            try:
                _, forecast     = fetch_city(client, city, code, delay_seconds)
                forecasts[city] = forecast
                print(f"  ✅ {city} — recovered on retry {retry_round}")
            except Exception as e:
                print(f"  ❌ {city} — still failing: {e}")
                still_failed[city] = str(e)

        failed       = still_failed
        retry_delay *= 2

    # ── Step 3: Top-up pass — catch anything still missing ────────────────────
    # Sometimes AEMET returns empty forecast (not an error) on first call
    # This pass specifically targets cities in forecasts but with empty data
    empty_cities = {
        city: verified_cities[city]
        for city, forecast in forecasts.items()
        if not forecast
    }
    if empty_cities:
        print(f"\n🔁 Top-up pass for {len(empty_cities)} cities with empty forecasts...")
        time.sleep(3)
        for city, code in empty_cities.items():
            try:
                _, forecast     = fetch_city(client, city, code, 1.0)
                forecasts[city] = forecast
                print(f"  ✅ {city} — filled on top-up pass")
            except Exception as e:
                print(f"  ❌ {city} — top-up failed: {e}")
                failed[city] = str(e)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  ✅ Fetched     : {len(forecasts)}/{len(SPAIN_CITIES)} cities")
    if failed:
        print(f"  ❌ Failed      : {list(failed.keys())}")
        for city, err in failed.items():
            print(f"     {city}: {err}")
    else:
        print(f"  🎉 All cities fetched successfully!")
    print(f"{'='*60}")

    return forecasts, fetch_time, list(failed.keys())


# ── Run ───────────────────────────────────────────────────────────────────────
all_forecasts, fetch_time, permanently_failed = fetch_all_spain(
    AEMET_API_KEY,
    delay_seconds = 0.5,
    max_retries   = 3,
    retry_delay   = 5.0
)

# ── Liveness verification ─────────────────────────────────────────────────────
madrid_raw         = AEMETClient(AEMET_API_KEY).get_weather_by_city("28079")
madrid_elaborated  = madrid_raw.get("elaborado")
madrid_next_update = madrid_raw.get("proximosDatos")

first_forecast_date = all_forecasts["Madrid"][0]["date"] if all_forecasts.get("Madrid") else None
today    = date.today().isoformat()
tomorrow = date.fromordinal(date.today().toordinal() + 1).isoformat()
is_live  = first_forecast_date in [today, tomorrow]

print(f"\n  Fetched at (UTC)    : {fetch_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  AEMET elaborated at : {madrid_elaborated}")
print(f"  Next AEMET update   : {madrid_next_update}")
print(f"  First forecast date : {first_forecast_date}  "
      f"({'✅ Live' if is_live else '⚠️ Check — may be stale'})")
print(f"  Permanently failed  : {permanently_failed if permanently_failed else 'None 🎉'}")
print(f"  Total coverage      : {len(all_forecasts)}/52 cities")

# ── Forecast printout ─────────────────────────────────────────────────────────
for city, forecast in all_forecasts.items():
    print(f"\n=== {city} ===")
    for d in forecast:
        sky = ", ".join(d["sky_conditions"]) if d["sky_conditions"] else "N/A"
        print(f"  {d['date']} | {d['temp_min']}°C – {d['temp_max']}°C | {sky}")

In [ ]:
import time

def recover_missing_cities(
    all_forecasts: dict,
    spain_cities:  dict,
    api_key:       str,
    max_rounds:    int   = 5,
    round_wait:    float = 65,   # seconds between rounds (> 60 to reset AEMET limit)
    request_wait:  float = 2     # seconds between individual requests
) -> dict:
    """
    Keeps fetching missing cities until all are recovered or max_rounds reached.
    Returns the updated all_forecasts dict.
    """
    client = AEMETClient(api_key)

    for round_num in range(1, max_rounds + 1):

        # ── Check what's still missing ────────────────────────────────────────
        missing = {
            city: code
            for city, code in spain_cities.items()
            if city not in all_forecasts or not all_forecasts.get(city)
        }

        if not missing:
            print(f"🎉 All {len(spain_cities)} cities fetched successfully!")
            break

        print(f"\n{'='*55}")
        print(f"  Recovery round   : {round_num}/{max_rounds}")
        print(f"  Missing cities   : {list(missing.keys())}")
        print(f"  Waiting {round_wait}s for AEMET rate limit to reset...")
        print(f"{'='*55}\n")
        time.sleep(round_wait)

        recovered_this_round = []
        still_missing        = []

        for city, code in missing.items():
            for attempt in range(1, 4):   # 3 attempts per city
                try:
                    time.sleep(request_wait)
                    raw      = client.get_weather_by_city(code)
                    forecast = get_forecast_summary(raw)
                    if not forecast:
                        raise ValueError("Empty forecast returned")
                    all_forecasts[city] = forecast
                    recovered_this_round.append(city)
                    print(f"  ✅ {city} recovered")
                    break
                except Exception as e:
                    if attempt < 3:
                        print(f"  ⏳ {city} attempt {attempt} failed ({e}) — retrying in 5s...")
                        time.sleep(5)
                    else:
                        print(f"  ❌ {city} failed this round: {e}")
                        still_missing.append(city)

        # ── Round summary ─────────────────────────────────────────────────────
        print(f"\n  Round {round_num} results:")
        print(f"  ✅ Recovered     : {recovered_this_round if recovered_this_round else 'none'}")
        print(f"  ❌ Still missing : {still_missing if still_missing else 'none'}")
        print(f"  📊 Coverage      : {len(all_forecasts)}/{len(spain_cities)}")

        # If no progress, double the wait time for next round (cap at 5 min)
        if not recovered_this_round:
            round_wait = min(round_wait * 2, 300)
            print(f"  ⚠️  No progress — increasing wait to {round_wait}s next round")

    return all_forecasts


# ── Run recovery ──────────────────────────────────────────────────────────────
all_forecasts = recover_missing_cities(
    all_forecasts = all_forecasts,
    spain_cities  = SPAIN_CITIES,
    api_key       = AEMET_API_KEY
)

# ── Final status ──────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Final coverage: {len(all_forecasts)}/52 cities")
missing_final = set(SPAIN_CITIES.keys()) - set(all_forecasts.keys())
if missing_final:
    print(f"  Still missing : {sorted(missing_final)}")
else:
    print(f"  Status        : All 52 cities ready ✅")
print(f"{'='*55}")

---
## 📄 PHASE 2 — RAG over AEMET Alert Documents

### 6. Load & Chunk PDF Documents

In [ ]:
import shutil
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -------------------------------------------------------
# ✏️  UPDATE this path to wherever your AEMET PDFs live
#   • Google Colab:  "/content/aemet_pdfs"
#   • Local:         r"F:\Projects\AEMET_Project"
# -------------------------------------------------------
PDF_DIR = r"F:\Projects\AEMET _Project\data"

all_docs = []
for filename in os.listdir(PDF_DIR):
    if filename.endswith(".pdf"):
        path = os.path.join(PDF_DIR, filename)
        print(f"Loading: {filename}")
        loader = PyPDFLoader(path)
        all_docs.extend(loader.load())

print(f"\nTotal pages loaded: {len(all_docs)}")

# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks   = splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

### 7. Embed & Store in ChromaDB

In [ ]:
from langchain_chroma import Chroma          # ← replaces langchain_community version
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb
import time
import os
import gc

os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

COLLECTION_NAME   = "AEMET_alerts"
PERSIST_DIRECTORY = "./chroma_db"

# ── Load embedding model ──────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"token": HF_TOKEN}
)
print("Embedding model loaded ✅")

# ── Release any existing vectorstore handle (Windows file-lock fix) ───────────
try:
    vectorstore
    vectorstore._client = None
    del vectorstore
    gc.collect()
    time.sleep(1)
    print("Released old vectorstore handle ✅")
except NameError:
    pass

# ── Delete old collection via ChromaDB API ────────────────────────────────────
try:
    chroma_client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)
    existing      = [c.name for c in chroma_client.list_collections()]
    if COLLECTION_NAME in existing:
        chroma_client.delete_collection(COLLECTION_NAME)
        print(f"Deleted old collection '{COLLECTION_NAME}' ✅")
    else:
        print("No existing collection — creating fresh ✅")
    del chroma_client
    gc.collect()
    time.sleep(0.5)
except Exception as e:
    print(f"Could not delete collection (will overwrite): {e}")

# ── Store chunks in ChromaDB ──────────────────────────────────────────────────
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY
)
print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB ✅")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready ✅")

---
### 🔗 UNIFIED CHAIN — Live Forecast + RAG Context → LLM

### 8. Initialise ChatGroq LLM

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=1024
)
print("ChatGroq LLM ready ✅")

In [ ]:
from langdetect import detect
import time
from langchain_core.messages import SystemMessage, HumanMessage

# Cities used as a fallback when the question doesn't name a specific city.
# Keeps the prompt small instead of dumping all 52 cities (this was the
# main cause of the token-limit errors).
FALLBACK_CITIES = ["Madrid", "Barcelona", "Sevilla", "Valencia",
                    "Bilbao", "Málaga", "Zaragoza", "Murcia"]

MAX_RAG_CHARS_PER_CHUNK = 300   # truncate each retrieved chunk
MAX_FORECAST_DAYS       = 1     # only "today" instead of 2 days


def filter_relevant_cities(question, forecast_data, max_fallback_cities=FALLBACK_CITIES):
    """
    Return only cities mentioned in the question.
    If no city is detected, return a small curated subset instead of
    every city — sending all 52 cities was blowing past Groq's TPM limit.
    """
    question_lower = question.lower()

    matches = {
        city: days
        for city, days in forecast_data.items()
        if city.lower() in question_lower
    }

    if matches:
        return matches

    return {
        city: forecast_data[city]
        for city in max_fallback_cities
        if city in forecast_data
    }


def run_unified_chain(
    question: str,
    forecast_data: dict,
    retriever,
    llm,
    verbose: bool = True
) -> str:

    # ─────────────────────────────────────────────────────────────
    # Detect question language
    # ─────────────────────────────────────────────────────────────
    try:
        detected_language = detect(question)
    except Exception:
        detected_language = "en"

    # ─────────────────────────────────────────────────────────────
    # Retrieve RAG context (k is set on the retriever itself, see
    # cell 20 — lowered from 5 to 3 chunks to save tokens)
    # ─────────────────────────────────────────────────────────────
    retrieved_docs = retriever.invoke(question)

    rag_context = "\n\n".join(
        doc.page_content[:MAX_RAG_CHARS_PER_CHUNK]
        for doc in retrieved_docs
    )

    if verbose:
        print(
            f"[RAG] Retrieved "
            f"{len(retrieved_docs)} chunks from ChromaDB"
        )

    # ─────────────────────────────────────────────────────────────
    # Filter forecast data (capped fallback — see filter_relevant_cities)
    # ─────────────────────────────────────────────────────────────
    relevant_forecasts = filter_relevant_cities(
        question,
        forecast_data
    )

    # ─────────────────────────────────────────────────────────────
    # Format forecast context (compact: 1 day, single rain/storm
    # number instead of full period-by-period breakdown)
    # ─────────────────────────────────────────────────────────────
    forecast_text = ""

    for city, days in relevant_forecasts.items():

        forecast_text += (
            f"\n### {city.upper()} FORECAST\n"
        )

        for d in days[:MAX_FORECAST_DAYS]:

            sky = (
                ", ".join(d["sky_conditions"])
                if d["sky_conditions"]
                else "N/A"
            )

            rain = d["rain_probability"][0] if d["rain_probability"] else "N/A"
            storm = d["storm_probability"][0] if d["storm_probability"] else "N/A"

            forecast_text += (
                f"{d['date']}: "
                f"{d['temp_min']}°C – "
                f"{d['temp_max']}°C | "
                f"Sky: {sky} | "
                f"Rain: {rain} | "
                f"Storm: {storm}\n"
            )

    # ─────────────────────────────────────────────────────────────
    # Build prompt (trimmed boilerplate)
    # ─────────────────────────────────────────────────────────────
    system_prompt = f"""You are a weather assistant for Spain (AEMET data). Question language: {detected_language}

[LIVE FORECAST]
{forecast_text}

[AEMET ALERT CONTEXT]
{rag_context if rag_context else "No relevant documents retrieved."}

Rules: use both sources when relevant; if info is unavailable say so; always answer in the language above; be concise."""

    # ─────────────────────────────────────────────────────────────
    # Token safety check before calling the LLM
    # ─────────────────────────────────────────────────────────────
    est_tokens = len(system_prompt + question) // 4
    if verbose:
        print(f"[Tokens] Estimated prompt tokens: ~{est_tokens}")
    if est_tokens > 5500:
        # Hard safety net: drop RAG context first, then trim forecast text
        if rag_context:
            system_prompt = system_prompt.replace(rag_context, "(omitted to stay under token limit)")
            if verbose:
                print("[Tokens] Over budget — dropped RAG context to stay under Groq's 6,000 TPM limit.")

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    # ─────────────────────────────────────────────────────────────
    # Retry logic for Groq rate limits
    # ─────────────────────────────────────────────────────────────
    for attempt in range(3):

        try:

            response = llm.invoke(messages)

            return response.content

        except Exception as e:

            error_text = str(e).lower()

            if (
                "rate limit" in error_text
                or "429" in error_text
                or "too many requests" in error_text
                or "413" in error_text
                or "too large" in error_text
            ):

                print(
                    f"[Retry {attempt+1}/3] "
                    f"Rate/size limit reached. "
                    f"Waiting 10 seconds..."
                )

                time.sleep(10)
                continue

            raise

    return (
        "Unable to generate a response "
        "after multiple retry attempts."
    )


print("run_unified_chain() defined ✅")


In [ ]:
question = (
    "Since I don't like hot summer weather, "
    "which city in Spain would you recommend "
    "for a holiday in the next few days?"
)

answer = run_unified_chain(
    question=question,
    forecast_data=all_forecasts,
    retriever=retriever,
    llm=llm
)

print(answer)

In [ ]:
# ── Inline token diagnostic ───────────────────────────────────────────────────
# (previously called an undefined format_all_forecasts() function — fixed to
# reuse the same filter_relevant_cities() + run_unified_chain() formatting
# logic so this diagnostic reflects what's actually sent to the LLM)
question        = "What is the weather in Madrid?"
relevant        = filter_relevant_cities(question, all_forecasts)
forecast_text   = ""
for city, days in relevant.items():
    forecast_text += f"\n### {city.upper()} FORECAST\n"
    for d in days[:MAX_FORECAST_DAYS]:
        sky = ", ".join(d["sky_conditions"]) if d["sky_conditions"] else "N/A"
        forecast_text += f"{d['date']}: {d['temp_min']}°C–{d['temp_max']}°C | Sky: {sky}\n"

retrieved_docs = retriever.invoke(question)
rag_context    = "\n\n".join(doc.page_content[:MAX_RAG_CHARS_PER_CHUNK] for doc in retrieved_docs)

system_prompt = f"""You are a concise weather assistant for Spain (AEMET data).
LANGUAGE RULE: Respond in English.
[LIVE FORECAST]
{forecast_text}
[AEMET ALERT DOCS]
{rag_context}
Rules: be concise."""

print(f"Forecast text chars  : {len(forecast_text)}")
print(f"RAG context chars    : {len(rag_context)}")
print(f"Full prompt chars    : {len(system_prompt + question)}")
print(f"Estimated tokens     : ~{len(system_prompt + question) // 4}")
print(f"Groq TPM limit       : 6,000")
print(f"Status               : {'❌ OVER LIMIT' if len(system_prompt + question) // 4 > 6000 else '✅ OK'}")
print()
print("── Forecast text preview (first 300 chars) ──")
print(forecast_text[:300])


---
### 💬 Interactive Query Loop

Ask any weather question. Type `exit` to quit.

In [ ]:
import time
import traceback
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Verify forecasts loaded ───────────────────────────────────────────────────
print(f"Cities available in all_forecasts: {len(all_forecasts)}")
print(f"City list: {list(all_forecasts.keys())}\n")

if len(all_forecasts) == 0:
    print("❌ all_forecasts is empty — re-run the fetch cell before starting the loop.")

else:
    print("🌤️  AEMET Unified Weather Assistant")
    print(f"   Coverage  : {len(all_forecasts)} Spanish provinces")
    print(f"   LLM model : {llm.model_name}")
    print(f"   RAG chunks: {vectorstore._collection.count()} in ChromaDB")
    print("   Type 'exit' / 'q' to quit | Leave blank + Enter to cancel current\n")

    COOLDOWN = 20

    def interruptible_sleep(seconds: int, label: str = "Cooldown") -> None:
        try:
            for remaining in range(seconds, 0, -1):
                print(f"\r⏳ {label}: {remaining}s remaining...  ", end="", flush=True)
                time.sleep(1)
            print(f"\r{'':60}", end="\r")
        except KeyboardInterrupt:
            print(f"\n⚡ {label} skipped.")

    # ── Widget UI ─────────────────────────────────────────────────────────────
    text_input = widgets.Text(
        placeholder="Ask a weather question...",
        layout=widgets.Layout(width="600px")
    )
    ask_btn    = widgets.Button(description="Ask",    button_style="primary")
    cancel_btn = widgets.Button(description="Cancel", button_style="warning")
    quit_btn   = widgets.Button(description="Quit",   button_style="danger")
    out        = widgets.Output()

    cancel_btn.disabled = True   # only active while a query runs
    running = {"flag": False}    # mutable flag accessible inside callbacks

    def run_query(question: str) -> None:
        """Runs inside the button callback — Output widget captures prints."""
        running["flag"] = True
        cancel_btn.disabled = False
        ask_btn.disabled    = True

        with out:
            print(f"\n❓ You: {question}")
            print("Thinking...\n")
            start = time.time()
            try:
                answer = run_unified_chain(
                    question      = question,
                    forecast_data = all_forecasts,
                    retriever     = retriever,
                    llm           = llm,
                    verbose       = True,
                )
                elapsed = time.time() - start
                print(f"\nAnswer:\n{answer}")
                print(f"\n⏱️  {elapsed:.2f}s")
                interruptible_sleep(COOLDOWN)

            except KeyboardInterrupt:
                print("\n⚡ Query interrupted.")

            except Exception as e:
                print(f"\n❌ {type(e).__name__}: {e}")
                traceback.print_exc()
                err_str = str(e)
                if "429" in err_str or "rate_limit" in err_str.lower():
                    print("\n⏳ Rate limit — waiting 60s...")
                    interruptible_sleep(60, label="Rate-limit wait")
                elif "413" in err_str or "too large" in err_str.lower():
                    print("\n📏 Prompt too large — try a more specific question.")

            finally:
                running["flag"]     = False
                cancel_btn.disabled = True
                ask_btn.disabled    = False
                text_input.value    = ""
                print("\n" + "-" * 70)

    def on_ask(b):
        question = text_input.value.strip()
        if not question:
            return
        run_query(question)

    def on_cancel(b):
        """Raises KeyboardInterrupt in the main thread — interrupts run_query."""
        import ctypes, threading
        ctypes.pythonapi.PyThreadState_SetAsyncExc(
            ctypes.c_ulong(threading.main_thread().ident),
            ctypes.py_object(KeyboardInterrupt),
        )

    def on_quit(b):
        with out:
            print("Goodbye! 👋")
        text_input.disabled = True
        ask_btn.disabled    = True
        cancel_btn.disabled = True
        quit_btn.disabled   = True

    # ── Wire up events ────────────────────────────────────────────────────────
    ask_btn.on_click(on_ask)
    cancel_btn.on_click(on_cancel)
    quit_btn.on_click(on_quit)
    text_input.on_submit(on_ask)   # Enter key also submits

    display(
        widgets.HBox([text_input, ask_btn, cancel_btn, quit_btn]),
        out
    )

In [ ]:
#some questions to ask
#1. Since I don't like hot summer weather, which city in Spain would you recommend for a holiday this week?
#2. Which cities in Spain currently have the most pleasant weather for tourists?
#3. I want to avoid rain during my holiday. Which Spanish city currently looks most suitable?
#4. Compare Madrid, Santander and Oviedo. Which would be best for sightseeing this week?
#5. Which city currently has the lowest weather risk for travelling, considering both forecasts and weather alerts?

## PHASE 3 — Conversational agent with memory

In [ ]:
from langchain_core.tools import tool
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.agents import create_agent
from langchain_groq import ChatGroq

### 9. Define LangChain Tools

In [ ]:
@tool
def get_live_weather(city: str) -> str:
    """Get the current AEMET forecast for a Spanish city. Input: city name e.g. 'Madrid'."""
    
    # Step 1: exact match (case-insensitive)
    matches = {c: d for c, d in all_forecasts.items() if c.lower() == city.lower()}
    
    # Step 2: partial match
    if not matches:
        city_lower = city.lower()
        matches = {c: d for c, d in all_forecasts.items() 
                   if city_lower in c.lower() or c.lower() in city_lower}
    
    # Step 3: nothing found
    if not matches:
        return f"No forecast data available for '{city}'."
    
    # Step 4: format output
    out = ""
    for c, days in matches.items():
        out += f"\n{c.upper()} FORECAST\n"
        for d in days[:2]:
            sky = ", ".join(d["sky_conditions"]) if d["sky_conditions"] else "N/A"
            out += f"{d['date']}: {d['temp_min']}°C–{d['temp_max']}°C | Sky: {sky}\n"
    return out

print("Tool 1 defined ✅")


In [ ]:
@tool
def search_alerts(query: str) -> str:
    """Search AEMET official documents for weather alert meanings, risk levels, and safety advice."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant AEMET alert documents found."
    out = ""
    for i, doc in enumerate(docs[:3], 1):
        content = doc.page_content[:400]
        content = content.replace("<", "").replace(">", "").replace("{", "").replace("}", "")
        out += f"[Doc {i}] {content}\n\n"
    return out

print("Tool 2 defined ✅")

In [ ]:
@tool
def compare_cities(cities: str) -> str:
    """Compare today's weather side by side for multiple Spanish cities. 
    Use when the user mentions TWO OR MORE cities, or uses words like 
    compare, difference, vs, versus. Input: comma-separated city names e.g. 'Madrid, Sevilla'."""
    city_list = [c.strip() for c in cities.split(",")]
    
    out = ""
    for city in city_list:
        # Step 1: exact match
        matches = {c: d for c, d in all_forecasts.items() if c.lower() == city.lower()}
        
        # Step 2: partial match
        if not matches:
            city_lower = city.lower()
            matches = {c: d for c, d in all_forecasts.items() 
                       if city_lower in c.lower() or c.lower() in city_lower}
        
        # Step 3: nothing found — skip, don't exit
        if not matches:
            out += f"{city}: No forecast data available.\n"
            continue
        
        # Step 4: first match, today only
        city_name = list(matches.keys())[0]
        today = matches[city_name][0]
        sky = ", ".join(today["sky_conditions"]) if today["sky_conditions"] else "N/A"
        out += f"{city_name}: {today['temp_min']}°C–{today['temp_max']}°C | Sky: {sky}\n"
    
    return out

print("Tool 3 defined ✅")

### 10. Build Agent with Memory

In [ ]:
agent_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=512
)

tools = [get_live_weather, search_alerts, compare_cities]

agent = create_agent(
    model=agent_llm,
    tools=tools,
    system_prompt=(
        "You are a weather assistant for Spain using AEMET data. "
        "Reply in the same language as the question (EN/ES). "
        "IMPORTANT: After receiving a tool result, you MUST immediately "
        "summarise it in plain text and stop. Do NOT call any tool more than once. "
        "TOOL SELECTION RULES:\n"
        "1. get_live_weather: ONE city forecast questions.\n"
        "2. compare_cities: TWO OR MORE cities, or words like compare/vs/versus.\n"
        "3. search_alerts: ONLY for alert meaning, risk levels, safety advice.\n"
    )
)

print("Agent defined ✅")

In [ ]:
chat_store = {}

def get_session_history(session_id):
    if session_id not in chat_store:
        chat_store[session_id] = InMemoryChatMessageHistory()
    return chat_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
)

print("Memory ready ✅")

### 11. Run Interactive Agent Loop

In [ ]:
print("🌤️  AEMET Weather Assistant — Phase 3")
print(f"   Model   : {agent_llm.model_name}")
print(f"   Cities  : {len(all_forecasts)} Spanish provinces")
print("   Type 'q' to quit\n")

config = {
    "configurable": {"session_id": "phase3_demo"},
    "recursion_limit": 10
}

while True:
    question = input("You: ").strip()
    if not question:
        continue
    if question.lower() in ["exit", "q", "quit"]:
        print("Goodbye! 👋")
        break
    try:
        result = agent_with_memory.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config
        )
        print(f"\nAssistant: {result['messages'][-1].content}\n")
        print("-" * 50)
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}\n")